In [1]:
# -*- coding: utf-8 -*-
"""
LFSR 100-step simulator with Tkinter buttons and final statistics.

How to run on Windows:
    python lfsr_100_tkinter_complete.py

Notes:
- This version does NOT start automatically. Press Play or Run 100.
- It avoids Japanese font dependency warnings by using English labels.
- Buttons are Tkinter buttons, not matplotlib.widgets.Button.
"""

import random
import tkinter as tk
from tkinter import ttk
import matplotlib

# Use TkAgg explicitly for Tkinter embedding.
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.patches import Rectangle

STEPS = 100
DEFAULT_INTERVAL_MS = 500
MAX_SHOW_BITS = 64

PRIMITIVE_POLYNOMIALS = {
    4:  [[4, 1, 0], [4, 3, 0]],
    5:  [[5, 2, 0], [5, 4, 2, 1, 0]],
    6:  [[6, 1, 0], [6, 5, 0]],
    7:  [[7, 1, 0], [7, 3, 0]],
    8:  [[8, 4, 3, 2, 0], [8, 6, 5, 4, 0]],
    10: [[10, 3, 0], [10, 7, 0]],
    12: [[12, 6, 4, 1, 0]],
    16: [[16, 12, 3, 1, 0], [16, 5, 3, 2, 0]],
    24: [[24, 4, 3, 1, 0]],
    32: [[32, 22, 2, 1, 0]],
    48: [[48, 28, 27, 1, 0]],
    64: [[64, 4, 3, 1, 0]],
}


def bits_to_str(bits):
    return "".join(str(b) for b in bits)


def polynomial_to_string(exponents):
    terms = []
    for e in sorted(exponents, reverse=True):
        if e == 0:
            terms.append("1")
        elif e == 1:
            terms.append("x")
        else:
            terms.append(f"x^{e}")
    return " + ".join(terms)


def choose_random_primitive_polynomial():
    degree = random.choice(list(PRIMITIVE_POLYNOMIALS.keys()))
    exponents = random.choice(PRIMITIVE_POLYNOMIALS[degree])
    return degree, exponents


def polynomial_to_taps(degree, exponents):
    """Convert polynomial exponents to register indices used for XOR feedback."""
    taps = []
    for e in exponents:
        if e == degree:
            continue
        if e == 0:
            taps.append(degree - 1)
        else:
            taps.append(degree - 1 - e)
    return sorted(set(taps))


def random_seed_bits(degree):
    while True:
        seed = [random.randint(0, 1) for _ in range(degree)]
        if any(seed):
            return seed


class LFSR:
    def __init__(self, seed, taps):
        if all(b == 0 for b in seed):
            raise ValueError("Seed must not be all zero.")
        self.reg = seed[:]
        self.taps = taps[:]
        self.degree = len(seed)

    def step(self):
        before = self.reg[:]
        tap_values = [before[t] for t in self.taps]
        feedback = 0
        for v in tap_values:
            feedback ^= v
        output_bit = before[-1]
        self.reg = [feedback] + before[:-1]
        return {
            "before": before,
            "after": self.reg[:],
            "tap_values": tap_values,
            "feedback": feedback,
            "output_bit": output_bit,
        }


def make_history(seed, taps, steps):
    lfsr = LFSR(seed, taps)
    states = [seed[:]]
    feedbacks = [None]
    outputs_by_clock = [None]
    tap_values_by_clock = [None]
    output_sequence = []

    for _ in range(steps):
        info = lfsr.step()
        states.append(info["after"])
        feedbacks.append(info["feedback"])
        outputs_by_clock.append(info["output_bit"])
        tap_values_by_clock.append(info["tap_values"])
        output_sequence.append(info["output_bit"])

    return states, feedbacks, outputs_by_clock, tap_values_by_clock, output_sequence


def run_lengths(bits):
    if not bits:
        return []
    lengths = []
    current = bits[0]
    count = 1
    for b in bits[1:]:
        if b == current:
            count += 1
        else:
            lengths.append((current, count))
            current = b
            count = 1
    lengths.append((current, count))
    return lengths


def compute_stats(states, feedbacks, outputs, output_sequence, seed, theoretical_period):
    fb = [x for x in feedbacks if x is not None]
    out = output_sequence[:]
    runs = run_lengths(out)

    first_return_clock = None
    for i in range(1, len(states)):
        if states[i] == seed:
            first_return_clock = i
            break

    unique_states = len({tuple(s) for s in states})
    max_run = max((length for _, length in runs), default=0)
    avg_run = sum(length for _, length in runs) / len(runs) if runs else 0.0

    return {
        "steps": len(out),
        "output_0": out.count(0),
        "output_1": out.count(1),
        "output_1_ratio": (out.count(1) / len(out)) if out else 0.0,
        "feedback_0": fb.count(0),
        "feedback_1": fb.count(1),
        "unique_states": unique_states,
        "first_return_clock": first_return_clock,
        "theoretical_period": theoretical_period,
        "run_count": len(runs),
        "max_run": max_run,
        "avg_run": avg_run,
    }


class LFSRApp:
    def __init__(self, root):
        self.root = root
        self.root.title("LFSR 100-step Simulator")
        self.root.geometry("1150x760")

        self.degree, self.exponents = choose_random_primitive_polynomial()
        self.taps = polynomial_to_taps(self.degree, self.exponents)
        self.seed = random_seed_bits(self.degree)
        self.theoretical_period = 2 ** self.degree - 1

        self.states, self.feedbacks, self.outputs, self.tap_values, self.output_sequence = make_history(
            self.seed, self.taps, STEPS
        )

        self.index = 0
        self.playing = False
        self.after_id = None
        self.interval_ms = DEFAULT_INTERVAL_MS

        self._build_ui()
        self.draw()
        self.update_info()

    def _build_ui(self):
        top = ttk.Frame(self.root, padding=8)
        top.pack(side=tk.TOP, fill=tk.X)

        self.info_var = tk.StringVar()
        info_label = ttk.Label(top, textvariable=self.info_var, font=("Consolas", 10))
        info_label.pack(side=tk.LEFT, fill=tk.X, expand=True)

        controls = ttk.Frame(self.root, padding=8)
        controls.pack(side=tk.TOP, fill=tk.X)

        ttk.Button(controls, text="Play", command=self.play).pack(side=tk.LEFT, padx=4)
        ttk.Button(controls, text="Pause", command=self.pause).pack(side=tk.LEFT, padx=4)
        ttk.Button(controls, text="Prev", command=self.prev_step).pack(side=tk.LEFT, padx=4)
        ttk.Button(controls, text="Next", command=self.next_step).pack(side=tk.LEFT, padx=4)
        ttk.Button(controls, text="Reset", command=self.reset).pack(side=tk.LEFT, padx=4)
        ttk.Button(controls, text="Run 100", command=self.run_to_end).pack(side=tk.LEFT, padx=4)
        ttk.Button(controls, text="Stats", command=self.show_stats).pack(side=tk.LEFT, padx=4)
        ttk.Button(controls, text="New", command=self.new_problem).pack(side=tk.LEFT, padx=4)
        ttk.Button(controls, text="Quit", command=self.close).pack(side=tk.LEFT, padx=4)

        ttk.Label(controls, text="  Speed(ms):").pack(side=tk.LEFT, padx=(20, 4))
        self.speed_var = tk.IntVar(value=self.interval_ms)
        speed = ttk.Scale(
            controls,
            from_=100,
            to=2000,
            orient=tk.HORIZONTAL,
            command=self.on_speed_change,
            length=180,
        )
        speed.set(self.interval_ms)
        speed.pack(side=tk.LEFT, padx=4)
        self.speed_label_var = tk.StringVar(value=str(self.interval_ms))
        ttk.Label(controls, textvariable=self.speed_label_var, width=5).pack(side=tk.LEFT)

        ttk.Label(controls, text="  Clock:").pack(side=tk.LEFT, padx=(20, 4))
        self.clock_var = tk.IntVar(value=0)
        self.clock_scale = ttk.Scale(
            controls,
            from_=0,
            to=STEPS,
            orient=tk.HORIZONTAL,
            command=self.on_clock_change,
            length=260,
        )
        self.clock_scale.pack(side=tk.LEFT, padx=4)
        self.clock_label_var = tk.StringVar(value="0")
        ttk.Label(controls, textvariable=self.clock_label_var, width=4).pack(side=tk.LEFT)

        self.fig, self.ax = plt.subplots(figsize=(11.0, 4.7))
        self.fig.subplots_adjust(left=0.04, right=0.98, top=0.86, bottom=0.18)
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.root)
        self.canvas_widget = self.canvas.get_tk_widget()
        self.canvas_widget.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        bottom = ttk.Frame(self.root, padding=8)
        bottom.pack(side=tk.BOTTOM, fill=tk.X)
        self.text = tk.Text(bottom, height=9, wrap=tk.WORD, font=("Consolas", 10))
        self.text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scroll = ttk.Scrollbar(bottom, command=self.text.yview)
        scroll.pack(side=tk.RIGHT, fill=tk.Y)
        self.text.configure(yscrollcommand=scroll.set)

        self.root.bind("<space>", lambda e: self.toggle_play())
        self.root.bind("<Left>", lambda e: self.prev_step())
        self.root.bind("<Right>", lambda e: self.next_step())
        self.root.bind("<Home>", lambda e: self.reset())
        self.root.bind("<End>", lambda e: self.run_to_end())
        self.root.protocol("WM_DELETE_WINDOW", self.close)

    def on_speed_change(self, value):
        self.interval_ms = int(float(value))
        self.speed_label_var.set(str(self.interval_ms))

    def on_clock_change(self, value):
        # Avoid too many redraws while Play is running.
        if self.playing:
            return
        idx = int(round(float(value)))
        if idx != self.index:
            self.set_index(idx)

    def set_index(self, idx):
        self.index = max(0, min(int(idx), STEPS))
        self.clock_scale.set(self.index)
        self.clock_label_var.set(str(self.index))
        self.draw()
        self.update_info()

    def draw(self):
        self.ax.clear()
        reg = self.states[self.index]
        shown = reg[:MAX_SHOW_BITS]
        shown_n = len(shown)
        note = f"showing first {shown_n} of {len(reg)} bits" if len(reg) > shown_n else f"{len(reg)} bits"

        for i, bit in enumerate(shown):
            face = "#BFBFBF" if bit == 1 else "white"
            rect = Rectangle((i, 0), 1, 1, facecolor=face, edgecolor="black", linewidth=1.6)
            self.ax.add_patch(rect)
            self.ax.text(i + 0.5, 0.70, f"R{len(reg)-1-i}", ha="center", va="center", fontsize=8, fontweight="bold")
            self.ax.text(i + 0.5, 0.30, str(bit), ha="center", va="center", fontsize=16, fontweight="bold")

            if i in self.taps:
                self.ax.plot(i + 0.5, 1.12, marker="o", markersize=9, color="red")
                self.ax.text(i + 0.5, 1.34, "tap", ha="center", fontsize=8, color="red")

        feedback = self.feedbacks[self.index]
        output_bit = self.outputs[self.index]
        tap_vals = self.tap_values[self.index]

        info_parts = []
        if feedback is not None:
            info_parts.append(f"feedback={feedback}")
        if output_bit is not None:
            info_parts.append(f"output={output_bit}")
        if tap_vals is not None:
            info_parts.append("tap XOR: " + " xor ".join(map(str, tap_vals)) + f" = {feedback}")

        if info_parts:
            self.ax.text(shown_n / 2, 1.68, "    ".join(info_parts), ha="center", fontsize=11)
        else:
            self.ax.text(shown_n / 2, 1.68, "Initial state. Press Play, Next, or Run 100.", ha="center", fontsize=11)

        title = f"LFSR clock {self.index}/{STEPS}    G(x) = {polynomial_to_string(self.exponents)}    ({note})"
        self.ax.set_title(title, fontsize=12)
        self.ax.text(
            shown_n / 2,
            -0.35,
            "1 = gray, 0 = white. Feedback enters from the left; R0 exits from the right.",
            ha="center",
            fontsize=10,
        )

        margin = max(1.0, shown_n * 0.05)
        self.ax.set_xlim(-margin, shown_n + margin)
        self.ax.set_ylim(-0.65, 1.95)
        self.ax.axis("off")
        self.canvas.draw_idle()

    def update_info(self):
        self.info_var.set(
            f"degree={self.degree} | G(x)={polynomial_to_string(self.exponents)} | "
            f"taps={self.taps} | seed={bits_to_str(self.seed)} | "
            f"theoretical period=2^{self.degree}-1={self.theoretical_period} | steps={STEPS}"
        )
        if self.index == 0:
            msg = "Initial state. Press Play / Next / Run 100.\n"
        else:
            before = bits_to_str(self.states[self.index - 1])
            after = bits_to_str(self.states[self.index])
            msg = (
                f"Clock {self.index}\n"
                f"before : {before}\n"
                f"after  : {after}\n"
                f"output : {self.outputs[self.index]}\n"
                f"feedback: {self.feedbacks[self.index]}\n"
            )
        self.text.delete("1.0", tk.END)
        self.text.insert(tk.END, msg)
        if self.index == STEPS:
            self.text.insert(tk.END, "\nFinished 100 steps. Statistics are shown below.\n\n")
            self.text.insert(tk.END, self.stats_text())

    def play(self):
        if self.index >= STEPS:
            self.set_index(0)
        self.playing = True
        self.schedule_next()

    def pause(self):
        self.playing = False
        if self.after_id is not None:
            self.root.after_cancel(self.after_id)
            self.after_id = None

    def toggle_play(self):
        if self.playing:
            self.pause()
        else:
            self.play()

    def schedule_next(self):
        if not self.playing:
            return
        if self.after_id is not None:
            self.root.after_cancel(self.after_id)
        self.after_id = self.root.after(self.interval_ms, self.play_tick)

    def play_tick(self):
        self.after_id = None
        if not self.playing:
            return
        if self.index < STEPS:
            self.set_index(self.index + 1)
            self.schedule_next()
        else:
            self.pause()
            self.show_stats()

    def next_step(self):
        self.pause()
        self.set_index(self.index + 1)
        if self.index == STEPS:
            self.show_stats()

    def prev_step(self):
        self.pause()
        self.set_index(self.index - 1)

    def reset(self):
        self.pause()
        self.set_index(0)

    def run_to_end(self):
        self.pause()
        self.set_index(STEPS)
        self.show_stats()

    def stats_text(self):
        stats = compute_stats(
            self.states,
            self.feedbacks,
            self.outputs,
            self.output_sequence,
            self.seed,
            self.theoretical_period,
        )
        first_return = stats["first_return_clock"]
        first_return_text = str(first_return) if first_return is not None else "not observed in 100 steps"
        out_seq = bits_to_str(self.output_sequence)
        if len(out_seq) > 120:
            out_seq_display = out_seq[:120] + "..."
        else:
            out_seq_display = out_seq

        return (
            "Statistics\n"
            "----------\n"
            f"steps                  : {stats['steps']}\n"
            f"output 0 count          : {stats['output_0']}\n"
            f"output 1 count          : {stats['output_1']}\n"
            f"output 1 ratio          : {stats['output_1_ratio']:.3f}\n"
            f"feedback 0 count        : {stats['feedback_0']}\n"
            f"feedback 1 count        : {stats['feedback_1']}\n"
            f"unique states observed  : {stats['unique_states']}\n"
            f"first return to seed    : {first_return_text}\n"
            f"theoretical period      : {stats['theoretical_period']}\n"
            f"run count               : {stats['run_count']}\n"
            f"max run length          : {stats['max_run']}\n"
            f"average run length      : {stats['avg_run']:.3f}\n"
            f"output sequence         : {out_seq_display}\n"
        )

    def show_stats(self):
        self.pause()
        self.text.delete("1.0", tk.END)
        self.text.insert(tk.END, self.stats_text())

    def new_problem(self):
        self.pause()
        self.degree, self.exponents = choose_random_primitive_polynomial()
        self.taps = polynomial_to_taps(self.degree, self.exponents)
        self.seed = random_seed_bits(self.degree)
        self.theoretical_period = 2 ** self.degree - 1
        self.states, self.feedbacks, self.outputs, self.tap_values, self.output_sequence = make_history(
            self.seed, self.taps, STEPS
        )
        self.set_index(0)

    def close(self):
        self.pause()
        plt.close(self.fig)
        self.root.destroy()


def main():
    root = tk.Tk()
    app = LFSRApp(root)
    root.mainloop()


if __name__ == "__main__":
    main()


Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\user\anaconda3\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_8004\251424783.py", line 276, in on_speed_change
    self.speed_label_var.set(str(self.interval_ms))
    ^^^^^^^^^^^^^^^^^^^^
AttributeError: 'LFSRApp' object has no attribute 'speed_label_var'
